In [0]:

import pandas as pd
import os

# 1. RUTA: Carpeta 'Datos' del repositorio
folder_path = "/Workspace/Users/andreyrangelcc@gmail.com/Proyecto-SQL-Caso-Estudio-Hotel-ARC/Datos"

# 2. ESQUEMA: Destino de las tablas
catalog_schema = "cursodataacademy.proyecto_hotel"

# Buscar archivos CSV en el repositorio
csv_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".csv")]

print(f"Se encontraron {len(csv_files)} archivos CSV:\n")

for csv_file in sorted(csv_files):
    file_path = os.path.join(folder_path, csv_file)
    table_name = os.path.splitext(csv_file)[0].lower()
    full_table_name = f"{catalog_schema}.{table_name}"

    # 3. Verificar si la tabla ya existe
    if spark.catalog.tableExists(full_table_name):
        print(f"✅ Tabla existente, se conserva: {full_table_name}")
        continue

    # 4. Manejo de separadores
    separador = ";" if "ticket" in csv_file.lower() else ","

    # 5. Leer el CSV
    df_pandas = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        sep=separador,
        skipinitialspace=True
    )

    # 6. Limpiar nombres de columnas
    df_pandas.columns = df_pandas.columns.str.replace(
        r'[ ,;{}()\n\t=]', '_', regex=True
    )

    # 7. Crear DataFrame de Spark
    df_spark = spark.createDataFrame(df_pandas)

    # 8. Crear tabla Delta solo si no existe
    df_spark.write.mode("errorifexists").saveAsTable(full_table_name)

    row_count = df_spark.count()
    col_count = len(df_spark.columns)

    print(
        f" {csv_file} → {full_table_name} "
        f"({row_count} filas, {col_count} columnas: {df_spark.columns})"
    )

print(f"\n Verificación y carga finalizada en {catalog_schema}")
